In [1]:
import torch
from yolox.exp import get_exp
import os

# --- Configuration ---
# Path to your YOLOX checkpoint file (.pth)
ckpt_path = "YOLOX_outputs/yolox_nano_cid/best_ckpt.pth"

# Path to your YOLOX experiment file (e.g., yolox_nano_cid.py).
# This file defines the model architecture.
# Make sure this path is correct relative to where you run the script,
# or provide an absolute path.
exp_file = "exps/default/yolox_nano_cid.py" # <--- IMPORTANT: Update this to your actual experiment file

# --- Check for file existence ---
if not os.path.exists(ckpt_path):
    print(f"Error: Checkpoint file not found at '{ckpt_path}'")
    print("Please ensure the 'models' directory exists and 'best_ckpt.pth' is inside it.")
    exit()

if not os.path.exists(exp_file):
    print(f"Error: Experiment file not found at '{exp_file}'")
    print("Please ensure the 'exps' directory exists and your experiment file is inside it.")
    print("You might need to adjust 'exp_file' variable to point to your specific YOLOX experiment configuration.")
    exit()

print(f"Loading model from checkpoint: {ckpt_path}")
print(f"Using experiment file: {exp_file}")

try:
    # 1. Load YOLOX model architecture from the experiment file
    # The second argument (name) is usually None when loading directly from exp_file.
    exp = get_exp(exp_file, None)
    print(f"\nExperiment loaded successfully. Number of classes: {exp.num_classes}")

    # Get the model instance based on the experiment configuration
    model = exp.get_model()
    print(f"Model architecture obtained from experiment.")

    # 2. Load checkpoint
    # Use map_location="cpu" to load the model onto the CPU, regardless of where it was saved.
    # weights_only=False is important if the checkpoint contains more than just model weights
    # (e.g., optimizer state, epoch info).
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    # The actual model weights are usually under the 'model' key in YOLOX checkpoints.
    if "model" in ckpt:
        model.load_state_dict(ckpt["model"])
        print("Model weights loaded from checkpoint.")
    else:
        # If 'model' key is not found, try loading directly (less common for YOLOX)
        model.load_state_dict(ckpt)
        print("Model weights loaded directly (no 'model' key found in checkpoint).")

    # 3. Set the model to evaluation mode
    # This is crucial for inference, as it disables dropout and batch normalization updates.
    model.eval()
    print("Model set to evaluation mode.")

    # 4. Print the model's structure
    # This will display a detailed breakdown of all layers and their parameters.
    print("\n--- YOLOX Model Structure ---")
    print(model)
    print("\n--- End of Model Structure ---")

    # Optional: Print a summary of the model's layers and parameter count
    print("\n--- Model Summary ---")
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

except Exception as e:
    print(f"\nAn error occurred: {e}")
    print("Please ensure:")
    print("1. You have the YOLOX repository correctly installed and accessible.")
    print("2. The 'exp_file' variable points to the correct experiment configuration file for your model.")
    print("3. The 'ckpt_path' variable points to your actual checkpoint file.")
    print("4. Your PyTorch and YOLOX installations are compatible.")



Loading model from checkpoint: YOLOX_outputs/yolox_nano_cid/best_ckpt.pth
Using experiment file: exps/default/yolox_nano_cid.py

Experiment loaded successfully. Number of classes: 2
Model architecture obtained from experiment.
Model weights loaded from checkpoint.
Model set to evaluation mode.

--- YOLOX Model Structure ---
YOLOX(
  (backbone): YOLOPAFPN(
    (backbone): CSPDarknet(
      (stem): Focus(
        (conv): BaseConv(
          (conv): Conv2d(12, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
      )
      (dark2): Sequential(
        (0): DWConv(
          (dconv): BaseConv(
            (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
            (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
            (act): SiLU(inplace=Tru

In [2]:
import torch
from yolox.exp import get_exp
import os

# --- Configuration ---
# Path to your YOLOX checkpoint file (.pth)
# Ensure this path is correct.
ckpt_path = "YOLOX_outputs/yolox_nano_cid/best_ckpt.pth"

# Path to your YOLOX experiment file (e.g., yolox_nano_cid.py).
# This file defines the model architecture.
# IMPORTANT: Update this to the correct path of your experiment file.
# Based on your previous output, it seems to be 'exps/default/yolox_nano_cid.py'.
exp_file = "exps/default/yolox_nano_cid.py"

# Path for the output ONNX model.
onnx_path = "models/yolox_nano_cid.onnx"

# Input image dimensions (Height, Width) - must match your training input size.
# Based on your previous tensor details, 320x320 is used.
input_height = 320
input_width = 320

# --- Pre-checks ---
if not os.path.exists(ckpt_path):
    print(f"Error: Checkpoint file not found at '{ckpt_path}'")
    print("Please ensure the 'YOLOX_outputs/yolox_nano_cid/' directory exists and 'best_ckpt.pth' is inside it.")
    exit()

if not os.path.exists(exp_file):
    print(f"Error: Experiment file not found at '{exp_file}'")
    print("Please ensure the 'exps/default/' directory exists and your experiment file is inside it.")
    print("You might need to adjust 'exp_file' variable to point to your specific YOLOX experiment configuration.")
    exit()

# Create the 'models' directory if it doesn't exist
os.makedirs(os.path.dirname(onnx_path), exist_ok=True)

print(f"Starting YOLOX .pth to ONNX conversion...")
print(f"Loading model from checkpoint: {ckpt_path}")
print(f"Using experiment file: {exp_file}")
print(f"Output ONNX path: {onnx_path}")
print(f"Expected input size: {input_width}x{input_height}")

try:
    # 1. Load YOLOX model architecture from the experiment file
    # The second argument (name) is usually None when loading directly from exp_file.
    exp = get_exp(exp_file, None)
    print(f"Experiment loaded successfully. Number of classes: {exp.num_classes}")

    # Get the model instance based on the experiment configuration
    model = exp.get_model()
    print(f"Model architecture obtained from experiment.")

    # 2. Load checkpoint weights
    # Use map_location="cpu" to load the model onto the CPU, regardless of where it was saved.
    # weights_only=False is important if the checkpoint contains more than just model weights
    # (e.g., optimizer state, epoch info).
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    # YOLOX checkpoints typically store model weights under the 'model' key.
    if "model" in ckpt:
        model.load_state_dict(ckpt["model"])
        print("Model weights loaded from checkpoint.")
    else:
        # Fallback if the 'model' key is not present (less common for YOLOX)
        model.load_state_dict(ckpt)
        print("Model weights loaded directly (no 'model' key found in checkpoint).")

    # 3. Set the model to evaluation mode
    # This is crucial for inference, as it disables dropout and batch normalization updates.
    model.eval()
    print("Model set to evaluation mode.")

    # 4. Create a dummy input tensor
    # YOLOX models expect NCHW format (Batch, Channels, Height, Width).
    # The batch size is 1 for a single inference.
    dummy_input = torch.randn(1, 3, input_height, input_width)
    print(f"Created dummy input tensor with shape: {dummy_input.shape}")

    # 5. Export to ONNX
    # opset_version=11 is a commonly supported version.
    # dynamic_axes allows the batch size to be variable in the ONNX model.
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        input_names=["input"],   # Name for the input tensor in ONNX
        output_names=["output"], # Name for the output tensor in ONNX
        opset_version=11,
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        verbose=False, # Set to True for more detailed export logs
    )

    print(f"\nSuccessfully exported YOLOX model to ONNX at: {onnx_path}")

except Exception as e:
    print(f"\nAn error occurred during ONNX export: {e}")
    print("Please ensure:")
    print("1. You have the YOLOX repository correctly installed and accessible.")
    print("2. The 'exp_file' variable points to the correct experiment configuration file for your model.")
    print("3. The 'ckpt_path' variable points to your actual checkpoint file.")
    print("4. Your PyTorch and YOLOX installations are compatible.")
    print("5. The input_height and input_width match your model's expected input dimensions.")



Starting YOLOX .pth to ONNX conversion...
Loading model from checkpoint: YOLOX_outputs/yolox_nano_cid/best_ckpt.pth
Using experiment file: exps/default/yolox_nano_cid.py
Output ONNX path: models/yolox_nano_cid.onnx
Expected input size: 320x320
Experiment loaded successfully. Number of classes: 2
Model architecture obtained from experiment.
Model weights loaded from checkpoint.
Model set to evaluation mode.
Created dummy input tensor with shape: torch.Size([1, 3, 320, 320])

Successfully exported YOLOX model to ONNX at: models/yolox_nano_cid.onnx


In [10]:
import onnx
from onnx_tf.backend import prepare
import tensorflow as tf
import os

# --- Configuration ---
# Path to your input ONNX model.
# This should be the output path from the previous .pth to ONNX conversion script.
onnx_path = "models/yolox_nano_cid_raw_output.onnx"

# Path for the output TFLite model.
tflite_path = "models/yolox_nano_cid.tflite"

# --- Pre-checks ---
if not os.path.exists(onnx_path):
    print(f"Error: ONNX file not found at '{onnx_path}'")
    print("Please ensure the ONNX conversion from .pth was successful and the file exists.")
    exit()

# Create the 'models' directory if it doesn't exist
os.makedirs(os.path.dirname(tflite_path), exist_ok=True)

print(f"Starting ONNX to TFLite conversion...")
print(f"Input ONNX path: {onnx_path}")
print(f"Output TFLite path: {tflite_path}")

try:
    # 1. Load the ONNX model
    print("Loading ONNX model...")
    onnx_model = onnx.load(onnx_path)
    print("ONNX model loaded successfully.")

    # 2. Prepare the ONNX model for TensorFlow
    # This converts the ONNX graph to a TensorFlow graph and saves it as a SavedModel.
    print("Converting ONNX model to TensorFlow SavedModel...")
    tf_rep = prepare(onnx_model)
    
    # Define a temporary directory for the SavedModel
    saved_model_dir = "temp_tf_savedmodel"
    tf_rep.export_graph(saved_model_dir)
    print(f"TensorFlow SavedModel exported to: {saved_model_dir}")

    # 3. Load the TensorFlow SavedModel and convert to TFLite
    print("Converting TensorFlow SavedModel to TFLite...")
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)

    # Apply default optimizations (e.g., quantization if specified, graph optimizations)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # Enable Select TF Ops: This is crucial!
    # Some ONNX operations might be translated by onnx-tf into TensorFlow operations
    # that are not directly supported by TFLite's built-in ops.
    # Enabling SELECT_TF_OPS allows the TFLite runtime to use a TensorFlow ops delegate
    # to execute these operations. Without this, conversion might fail or the model
    # might not run on devices.
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,  # Enable standard TFLite operations
        tf.lite.OpsSet.SELECT_TF_OPS,    # Enable selected TensorFlow operations
    ]

    # Convert the model
    tflite_model = converter.convert()
    print("TFLite model conversion complete.")

    # 4. Save the TFLite model
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    print(f"\nSuccessfully exported TFLite model to: {tflite_path}")

except Exception as e:
    print(f"\nAn error occurred during ONNX to TFLite export: {e}")
    print("Please ensure:")
    print("1. The ONNX model at '{onnx_path}' is valid.")
    print("2. You have `onnx-tf` and `tensorflow` installed (`pip install onnx-tf tensorflow`).")
    print("3. The ONNX model does not contain operations that are fundamentally unsupported by TensorFlow Lite.")
    print("   (Often, complex post-processing like NMS is best handled client-side on mobile.)")

finally:
    # Clean up the temporary SavedModel directory
    if 'saved_model_dir' in locals() and os.path.exists(saved_model_dir):
        import shutil
        shutil.rmtree(saved_model_dir)
        print(f"Cleaned up temporary SavedModel directory: {saved_model_dir}")



Starting ONNX to TFLite conversion...
Input ONNX path: models/yolox_nano_cid_raw_output.onnx
Output TFLite path: models/yolox_nano_cid.tflite
Loading ONNX model...
ONNX model loaded successfully.
Converting ONNX model to TensorFlow SavedModel...


INFO:absl:Function `__call__` contains input name(s) x, y with unsupported characters which will be renamed to transpose_343_x, add_97_y in the SavedModel.
INFO:absl:Found untraced functions such as gen_tensor_dict while saving (showing 1 of 1). These functions will not be directly callable after loading.


INFO:tensorflow:Assets written to: temp_tf_savedmodel\assets


INFO:tensorflow:Assets written to: temp_tf_savedmodel\assets
INFO:absl:Writing fingerprint to temp_tf_savedmodel\fingerprint.pb


TensorFlow SavedModel exported to: temp_tf_savedmodel
Converting TensorFlow SavedModel to TFLite...
TFLite model conversion complete.

Successfully exported TFLite model to: models/yolox_nano_cid.tflite
Cleaned up temporary SavedModel directory: temp_tf_savedmodel


In [1]:
import tensorflow as tf
import numpy as np
import os

# --- Configuration ---
# Path to your TFLite model file.
tflite_model_path = "models/yolox_nano_cid.tflite"

# --- Pre-checks ---
if not os.path.exists(tflite_model_path):
    print(f"Error: TFLite model file not found at '{tflite_model_path}'")
    print("Please ensure the TFLite conversion was successful and the file exists.")
    exit()

print(f"Inspecting TFLite model: {tflite_model_path}")

try:
    # 1. Load the TFLite model and allocate tensors.
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()

    # 2. Get input tensor details.
    input_details = interpreter.get_input_details()
    print("\n--- Input Tensor Details ---")
    for i, detail in enumerate(input_details):
        print(f"Input Tensor {i}:")
        print(f"  Name: {detail['name']}")
        print(f"  Index: {detail['index']}")
        print(f"  Shape: {detail['shape']}")
        print(f"  Shape Signature: {detail['shape_signature']}")
        print(f"  Dtype: {detail['dtype']}")
        print(f"  Quantization: {detail['quantization']}")
        print(f"  Quantization Parameters: {detail['quantization_parameters']}")
        print(f"  Sparsity Parameters: {detail['sparsity_parameters']}")
        print("-" * 30)

    # 3. Get output tensor details.
    output_details = interpreter.get_output_details()
    print("\n--- Output Tensor Details ---")
    for i, detail in enumerate(output_details):
        print(f"Output Tensor {i}:")
        print(f"  Name: {detail['name']}")
        print(f"  Index: {detail['index']}")
        print(f"  Shape: {detail['shape']}")
        print(f"  Shape Signature: {detail['shape_signature']}")
        print(f"  Dtype: {detail['dtype']}")
        print(f"  Quantization: {detail['quantization']}")
        print(f"  Quantization Parameters: {detail['quantization_parameters']}")
        print(f"  Sparsity Parameters: {detail['sparsity_parameters']}")
        print("-" * 30)

    print("\nInspection complete.")

except Exception as e:
    print(f"\nAn error occurred during TFLite model inspection: {e}")
    print("Please ensure:")
    print("1. The TFLite model file at '{tflite_model_path}' is valid and not corrupted.")
    print("2. You have TensorFlow installed (`pip install tensorflow`).")




Inspecting TFLite model: models/yolox_nano_cid.tflite

--- Input Tensor Details ---
Input Tensor 0:
  Name: serving_default_input:0
  Index: 0
  Shape: [  1   3 320 320]
  Shape Signature: [ -1   3 320 320]
  Dtype: <class 'numpy.float32'>
  Quantization: (0.0, 0)
  Quantization Parameters: {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}
  Sparsity Parameters: {}
------------------------------

--- Output Tensor Details ---
Output Tensor 0:
  Name: PartitionedCall:0
  Index: 978
  Shape: [1 1 1]
  Shape Signature: [-1 -1 -1]
  Dtype: <class 'numpy.float32'>
  Quantization: (0.0, 0)
  Quantization Parameters: {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}
  Sparsity Parameters: {}
------------------------------

Inspection complete.


In [9]:
import torch
import torch.nn as nn
from yolox.exp import get_exp
import os

# --- Configuration ---
# Path to your YOLOX checkpoint file (.pth)
# Ensure this path is correct.
ckpt_path = "YOLOX_outputs/yolox_nano_cid/best_ckpt.pth"

# Path to your YOLOX experiment file (e.g., yolox_nano_cid.py).
# This file defines the model architecture.
# IMPORTANT: Update this to the correct path of your experiment file.
# Based on your previous output, it seems to be 'exps/default/yolox_nano_cid.py'.
exp_file = "exps/default/yolox_nano_cid.py"

# Path for the output ONNX model.
onnx_path = "models/yolox_nano_cid_raw_output.onnx"

# Input image dimensions (Height, Width) - must match your training input size.
# Based on your previous tensor details, 320x320 is used.
input_height = 320
input_width = 320

# Number of classes your model was trained on
num_classes = 2 # IMPORTANT: Set this to your actual number of classes

# --- Pre-checks ---
if not os.path.exists(ckpt_path):
    print(f"Error: Checkpoint file not found at '{ckpt_path}'")
    print("Please ensure the 'YOLOX_outputs/yolox_nano_cid/' directory exists and 'best_ckpt.pth' is inside it.")
    exit()

if not os.path.exists(exp_file):
    print(f"Error: Experiment file not found at '{exp_file}'")
    print("Please ensure the 'exps/default/' directory exists and your experiment file is inside it.")
    print("You might need to adjust 'exp_file' variable to point to your specific YOLOX experiment configuration.")
    exit()

# Create the 'models' directory if it doesn't exist
os.makedirs(os.path.dirname(onnx_path), exist_ok=True)

print(f"Starting YOLOX .pth to ONNX conversion...")
print(f"Loading model from checkpoint: {ckpt_path}")
print(f"Using experiment file: {exp_file}")
print(f"Output ONNX path: {onnx_path}")
print(f"Expected input size: {input_width}x{input_height}")

# --- Updated: Wrapper Module for ONNX Export with Fixed Output Shape ---
class ExportModel(nn.Module):
    def __init__(self, model, num_classes):
        super().__init__()
        self.backbone = model.backbone
        self.head = model.head
        self.num_classes = num_classes
        # Crucial: Ensure the head is configured for raw output during inference
        self.head.decode_in_inference = False

    def forward(self, x):
        # Pass input through the backbone to get feature maps
        fpn_outs = self.backbone(x)

        # Pass feature maps to the head to get raw predictions.
        # When self.head.decode_in_inference is False, the YOLOXHead typically
        # returns a single concatenated tensor of shape
        # [batch_size, (5 + num_classes), total_predictions].
        raw_output_tensor = self.head(fpn_outs)
        
        # Unconditionally permute the output to [batch_size, total_predictions, (5 + num_classes)].
        # This ensures a fixed, traceable output shape for ONNX.
        # We assume the raw_output_tensor is in NCL format.
        final_output = raw_output_tensor.permute(0, 2, 1)
            
        return final_output

try:
    # 1. Load YOLOX model architecture from the experiment file
    exp = get_exp(exp_file, None)
    print(f"Experiment loaded successfully. Number of classes: {exp.num_classes}")

    # Get the model instance based on the experiment configuration
    model = exp.get_model()
    print(f"Model architecture obtained from experiment.")

    # 2. Load checkpoint weights
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    if "model" in ckpt:
        model.load_state_dict(ckpt["model"])
        print("Model weights loaded from checkpoint.")
    else:
        model.load_state_dict(ckpt)
        print("Model weights loaded directly (no 'model' key found in checkpoint).")

    # 3. Set the original model to evaluation mode
    model.eval()
    print("Original model set to evaluation mode.")

    # 4. Instantiate the updated wrapper model
    export_model = ExportModel(model, num_classes=exp.num_classes) # Pass num_classes
    # Set the wrapper model to evaluation mode as well
    export_model.eval()
    print("Created ExportModel wrapper for ONNX conversion and set to evaluation mode.")

    # 5. Create a dummy input tensor
    # YOLOX models expect NCHW format (Batch, Channels, Height, Width).
    dummy_input = torch.randn(1, 3, input_height, input_width)
    print(f"Created dummy input tensor with shape: {dummy_input.shape}")

    # 6. Export to ONNX using the wrapper model
    # Now, the ExportModel.forward is expected to return a single tensor with a fixed shape.
    output_names = ["output"]
    print(f"Exporting with output names: {output_names}")

    # Calculate the expected fixed output shape for ONNX export.
    # This is for informational purposes and to confirm expectations.
    # Total predictions = (input_height/8 * input_width/8) + (input_height/16 * input_width/16) + (input_height/32 * input_width/32)
    total_predictions = (input_height // 8 * input_width // 8) + \
                        (input_height // 16 * input_width // 16) + \
                        (input_height // 32 * input_width // 32)
    attributes_per_prediction = 5 + exp.num_classes # 4 bbox + 1 obj + num_classes

    torch.onnx.export(
        export_model, # Use the wrapped model for export
        dummy_input,
        onnx_path,
        input_names=["input"],
        output_names=output_names,
        opset_version=11,
        # Define the fixed output dimensions for the ONNX graph.
        # The output shape should now be [batch_size, total_predictions, attributes_per_prediction].
        dynamic_axes={"input": {0: "batch_size"},
                      "output": {0: "batch_size"}}, # Only batch size is dynamic
        verbose=False, # Set to True for more detailed export logs
    )

    print(f"\nSuccessfully exported YOLOX model to ONNX at: {onnx_path}")

except Exception as e:
    print(f"\nAn error occurred during ONNX export: {e}")
    print("Please ensure:")
    print("1. You have the YOLOX repository correctly installed and accessible.")
    print("2. The 'exp_file' variable points to the correct experiment configuration file for your model.")
    print("3. The 'ckpt_path' variable points to your actual checkpoint file.")
    print("4. Your PyTorch and YOLOX installations are compatible.")
    print("5. The input_height and input_width match your model's expected input dimensions.")
    print("6. The 'num_classes' variable in the script matches your model's actual number of classes.")



Starting YOLOX .pth to ONNX conversion...
Loading model from checkpoint: YOLOX_outputs/yolox_nano_cid/best_ckpt.pth
Using experiment file: exps/default/yolox_nano_cid.py
Output ONNX path: models/yolox_nano_cid_raw_output.onnx
Expected input size: 320x320
Experiment loaded successfully. Number of classes: 2
Model architecture obtained from experiment.
Model weights loaded from checkpoint.
Original model set to evaluation mode.
Created ExportModel wrapper for ONNX conversion and set to evaluation mode.
Created dummy input tensor with shape: torch.Size([1, 3, 320, 320])
Exporting with output names: ['output']

Successfully exported YOLOX model to ONNX at: models/yolox_nano_cid_raw_output.onnx


In [11]:
import onnx
import numpy as np
import os

# --- Configuration ---
# Path to your ONNX model file.
# This should be the output from the .pth to ONNX conversion.
onnx_model_path = "models/yolox_nano_cid_raw_output.onnx"

# --- Pre-checks ---
if not os.path.exists(onnx_model_path):
    print(f"Error: ONNX model file not found at '{onnx_model_path}'")
    print("Please ensure the ONNX conversion was successful and the file exists.")
    exit()

print(f"Inspecting ONNX model: {onnx_model_path}")

try:
    # 1. Load the ONNX model
    model = onnx.load(onnx_model_path)
    graph = model.graph

    # 2. Get input tensor details
    print("\n--- ONNX Input Tensor Details ---")
    for input_node in graph.input:
        print(f"  Name: {input_node.name}")
        # Get shape from input_node.type.tensor_type.shape.dim
        shape = [dim.dim_value if dim.dim_value != 0 else -1 for dim in input_node.type.tensor_type.shape.dim]
        print(f"  Shape: {shape}")
        print(f"  Dtype: {onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[input_node.type.tensor_type.elem_type]}")
        print("-" * 30)

    # 3. Get output tensor details
    print("\n--- ONNX Output Tensor Details ---")
    for output_node in graph.output:
        print(f"  Name: {output_node.name}")
        # Get shape from output_node.type.tensor_type.shape.dim
        shape = [dim.dim_value if dim.dim_value != 0 else -1 for dim in output_node.type.tensor_type.shape.dim]
        print(f"  Shape: {shape}")
        print(f"  Dtype: {onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[output_node.type.tensor_type.elem_type]}")
        print("-" * 30)

    print("\nONNX model inspection complete.")

except Exception as e:
    print(f"\nAn error occurred during ONNX model inspection: {e}")
    print("Please ensure:")
    print("1. The ONNX model file at '{onnx_model_path}' is valid.")
    print("2. You have `onnx` installed (`pip install onnx`).")



Inspecting ONNX model: models/yolox_nano_cid_raw_output.onnx

--- ONNX Input Tensor Details ---
  Name: input
  Shape: [-1, 3, 320, 320]
  Dtype: float32
------------------------------

--- ONNX Output Tensor Details ---
  Name: output
  Shape: [-1, -1, -1]
  Dtype: float32
------------------------------

ONNX model inspection complete.


C:\Users\Wave\AppData\Local\Temp\ipykernel_24368\478887981.py:30: DeprecationWarning: `mapping.TENSOR_TYPE_TO_NP_TYPE` is now deprecated and will be removed in a future release.To silence this warning, please use `helper.tensor_dtype_to_np_dtype` instead.
  print(f"  Dtype: {onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[input_node.type.tensor_type.elem_type]}")
C:\Users\Wave\AppData\Local\Temp\ipykernel_24368\478887981.py:40: DeprecationWarning: `mapping.TENSOR_TYPE_TO_NP_TYPE` is now deprecated and will be removed in a future release.To silence this warning, please use `helper.tensor_dtype_to_np_dtype` instead.
  print(f"  Dtype: {onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[output_node.type.tensor_type.elem_type]}")
